# From return forecasts to portfolio decisions: a thesis tutorial

A hands-on companion to `backtest-light`, checked against its implementation on 2 November 2026.
Allow roughly 2–3 hours including derivations. No market data or network access is needed.

By the end you should be able to derive the allocation problem, explain how a CVXPYlayer propagates gradients, train a conditional Gaussian with a decision loss, and distinguish return uncertainty from a Bayesian posterior over an unknown mean.

**Workflow:** run in order, implement each TODO, then run its checks. `USE_SOLUTIONS=False` intentionally stops at unfinished exercises. To run the worked example, set it to `True` and restart/run all. Solution cells are tagged and marked collapsed (some notebook viewers display them anyway). All outputs are deliberately cleared in this exercise copy.

Use the repository's Python 3.10+ environment and existing `e2e` extra. From the repository root: `uv run --no-sync jupyter lab demo/ml/thesis_tutorial.ipynb`. If the extra is absent, install it yourself with `uv sync --extra e2e`; this notebook installs nothing.

**Scope:** the active trainer is point/Gaussian → convex allocation → realized decision loss. The standalone BL-style update is taught later as an optional extension, not a step currently used by `training/e2e.py`. This tutorial uses a fixed universe, diagonal risk, a smaller network and short training. It is a learning exercise, not empirical thesis evidence.

In [ ]:
from pathlib import Path
import sys
from copy import deepcopy
import math
import numpy as np
import torch
import cvxpy as cp
import matplotlib.pyplot as plt
from cvxpylayers.torch import CvxpyLayer

root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src/btlight").exists()), None)
if root is None:
    raise RuntimeError("Start Jupyter inside backtest-light")
sys.path.insert(0, str(root / "src"))
from btlight.ml.model.return_predictor import GaussianReturnPredictor, gaussian_nll
from btlight.optimization.differentiable import PortfolioLayer
from btlight.estimation.differentiable_bl import BLUpdate

USE_SOLUTIONS = False
torch.set_default_dtype(torch.float64)
torch.set_num_threads(1)
torch.manual_seed(7)
print("torch", torch.__version__, "cvxpy", cp.__version__)

## 1. Information, outcomes and shapes

At decision close $t$, observe $X_t\in\mathbb R^{N\times F}$. Allocate using only information then available. The real pipeline approximates execution at the next scheduled close and uses the return from that close to the following scheduled close as $y_t\in\mathbb R^N$. Here each simulated index represents one such complete interval. The network shares parameters across assets; it does not use asset IDs.

We simulate $y_{ti}=0.004x_{ti1}-0.002x_{ti2}+s_{ti}\epsilon_{ti}$, where $s_{ti}=0.008+0.004\operatorname{sigmoid}(x_{ti3})$ and independent $\epsilon_{ti}\sim N(0,1)$. The true simulator parameters are available **only for teaching checks**, never as network inputs. A separate fixed historical-risk proxy is supplied for scoring decisions. It is deliberately imperfect, like an estimated historical covariance.

Chronological splits prevent training on test outcomes. Real overlapping labels must also be purged by their end dates; simulated intervals here do not overlap.

In [ ]:
T, N, F = 64, 4, 3
generator = torch.Generator().manual_seed(31)
X = torch.randn(T, N, F, generator=generator)
true_mean = 0.004 * X[..., 0] - 0.002 * X[..., 1]
true_sd = 0.008 + 0.004 * torch.sigmoid(X[..., 2])
y = true_mean + true_sd * torch.randn(T, N, generator=generator)
reference_factor = torch.full((N,), 0.011)
train, validation, test = range(32), range(32, 48), range(48, 64)
assert X.shape == (T, N, F) and y.shape == (T, N)
assert max(train) < min(validation) < min(test)
plt.scatter(X[..., 0].flatten(), y.flatten(), s=8, alpha=.4)
plt.xlabel("Known feature 1"); plt.ylabel("Next interval return"); plt.show()

## 2. A conditional distribution, not just a point forecast

The network outputs $\mu_\theta(X_t)$ and positive marginal variances $v_\theta(X_t)$. This notebook uses
$$p_\theta(y_t\mid X_t)=N(\mu_t,\operatorname{diag}(v_t)),\qquad
v_i=v_{\min}+(v_0-v_{\min})\frac{\operatorname{softplus}(a_i)}{\log 2}.$$
At score $a_i=0$, the variance is $v_0$. Positive variance ensures a valid diagonal covariance. A full fixed-panel model can instead use $\Sigma_\theta=D_\theta R_{\rm hist}D_\theta$, where $D=\operatorname{diag}(\sqrt v)$ and $R$ is a historical correlation estimate. Changing-universe Gaussian training currently assumes diagonal covariance.

The per-asset negative log-likelihood is
$$\operatorname{NLL}_t=\frac1{2N}\sum_i\left[\log(2\pi)+\log v_{ti}+\frac{(y_{ti}-\mu_{ti})^2}{v_{ti}}\right].$$
It penalizes confident wrong predictions and excessive variance. Minimizing expected NLL targets the true conditional density if the model family can represent it; finite-sample training does not guarantee calibration. This is a single heteroscedastic network, not an ensemble or a posterior over network weights.

### Exercise: Gaussian NLL

Return one scalar averaged over assets, keeping gradients through both moments.

Implement `marginal_nll` below. The following cell contains a reference solution; try first before opening it.

In [ ]:
def marginal_nll(mu, variance, observed):
    # TODO: replace this exception with your implementation.
    raise NotImplementedError("Gaussian NLL")

In [ ]:
def reference_marginal_nll(mu, variance, observed):
    return 0.5 * (math.log(2 * math.pi) + variance.log()
                  + (observed - mu).square() / variance).mean()

if USE_SOLUTIONS:
    marginal_nll = reference_marginal_nll

In [ ]:
mu0 = torch.zeros(N, requires_grad=True)
v0 = torch.full((N,), 1e-4, requires_grad=True)
dist = torch.distributions.Independent(torch.distributions.Normal(mu0, v0.sqrt()), 1)
assert torch.allclose(marginal_nll(mu0, v0, y[0]), gaussian_nll(dist, y[0]))
gradients = torch.autograd.grad(marginal_nll(mu0, v0, y[0]), (mu0, v0))
assert all(torch.isfinite(g).all() for g in gradients)
print("NLL and moment-gradient checks passed")

## 3. The lower problem: choose weights

For forecasts $\mu$, risk factor $A$ with $A^\top A=\Sigma$, previous weights $w^-$, risk aversion $\lambda$ and cost rate $c$, solve
$$w^*=\arg\min_w\{-\mu^\top w+\lambda\|Aw\|_2^2+\tfrac c2\|w-w^-\|_1\},\quad
\mathbf1^\top w=1,\quad0\le w_i\le u.$$
Here $N=4$, $u=0.6$, $\lambda=20$, $c=0.0002$ are teaching choices. Feasibility requires $Nu\ge1$. The code has no extra $1/2$ on variance: its Hessian is $2\lambda\Sigma$. With positive definite covariance and $\lambda>0$ the objective is strictly convex and the solution is unique.

$\|Aw\|^2=w^\top\Sigma w$. If $L$ is a lower Cholesky factor ($\Sigma=LL^\top$), pass **$A=L^\top$**, not $L$. In the diagonal case pass the standard deviation vector and use elementwise multiplication. Portfolio allocation and decision loss use variance risk.

CVXPY declares a symbolic **variable** $w$ and runtime **parameters** $(\mu,A,w^-)$. DPP (disciplined parametrized programming) lets it compile the parameter-to-conic-data map once. Factorizing covariance outside CVXPY avoids the non-DPP parametrized quadratic form. DPP is a compilation rule, not a guarantee of smoothness or good conditioning.

CVXPYlayer returns a torch tensor with a backward rule; ordinary CVXPY `.solve()` and converting forecasts to NumPy break this training path.

### Exercise: Build the optimization layer

Use a diagonal factor parameter. Check DPP, then wrap the problem with parameter order `[mu, factor, previous]`.

Implement `build_layer` below. The following cell contains a reference solution; try first before opening it.

In [ ]:
def build_layer(assets, lam=20.0, cost=0.0002, cap=0.6):
    # TODO: replace this exception with your implementation.
    raise NotImplementedError("Build the optimization layer")

In [ ]:
def reference_build_layer(assets, lam=20.0, cost=0.0002, cap=0.6):
    w = cp.Variable(assets)
    mu = cp.Parameter(assets)
    factor = cp.Parameter(assets)
    previous = cp.Parameter(assets)
    objective = (-mu @ w + lam * cp.sum_squares(cp.multiply(factor, w))
                 + cost / 2 * cp.norm1(w - previous))
    problem = cp.Problem(cp.Minimize(objective),
                         [w >= 0, cp.sum(w) == 1, w <= cap])
    assert problem.is_dpp()
    return CvxpyLayer(problem, parameters=[mu, factor, previous], variables=[w])

if USE_SOLUTIONS:
    build_layer = reference_build_layer

In [ ]:
LAM, COST, CAP = 20.0, 0.0002, 0.6
solver_args = {"solve_method": "Clarabel", "tol_gap_abs": 1e-9,
               "tol_gap_rel": 1e-9, "tol_feas": 1e-9}
learner_layer = build_layer(N)
allocator = PortfolioLayer(N, LAM, CAP, cost=COST, diagonal=True)
previous0 = torch.full((N,), 1 / N)
probe_mu = torch.tensor([.0003, -.0001, .0001, -.0003])
w, = learner_layer(probe_mu, reference_factor, previous0, solver_args=solver_args)
w_repo = allocator(probe_mu, reference_factor, previous0)
assert torch.allclose(w, w_repo, atol=2e-6)
assert abs(w.sum().item() - 1) < 1e-6
assert w.min() > -1e-6 and w.max() < CAP + 1e-6
# A non-diagonal example catches the Cholesky orientation mistake.
Sigma = torch.tensor([[.0004, .0001], [.0001, .0002]])
L = torch.linalg.cholesky(Sigma)
z = torch.tensor([.3, .7])
assert torch.allclose((L.T @ z).square().sum(), z @ Sigma @ z)
print("Weights:", w.detach().numpy())

## 4. Why an argmin can have a derivative

First ignore turnover and bounds, keeping only the budget constraint. Stationarity gives
$$2\lambda\Sigma w-\mu+\nu\mathbf1=0,\quad\mathbf1^\top w=1.$$
Differentiate this system at a solution:
$$\begin{bmatrix}2\lambda\Sigma&\mathbf1\\\mathbf1^\top&0\end{bmatrix}
\begin{bmatrix}dw\\d\nu\end{bmatrix}
=\begin{bmatrix}d\mu-2\lambda(d\Sigma)w\\0\end{bmatrix}.$$
Thus changing a forecast changes the optimal decision. With bounds, a locally unchanged active set adds constraint rows; turnover can be represented with auxiliary variables in a conic program. CVXPYlayers differentiates the conic solution map implicitly, rather than backpropagating through every solver iteration. Backward computes vector–Jacobian products needed by the outer loss.

At active-set changes or L1 kinks a classical derivative may not exist. Numerical derivatives can also be unstable for poorly conditioned or nonunique problems. The next check deliberately uses zero turnover and an interior solution to test a smooth point.

**Derive on paper:** why does adding the same constant to all forecast means leave weights unchanged? Why must $\mathbf1^\top dw=0$?

### Exercise: Check a solver derivative

Compute d(w[0])/d(mu[0]) with autograd and a centered finite difference. Return both scalars.

Implement `gradient_check` below. The following cell contains a reference solution; try first before opening it.

In [ ]:
def gradient_check(layer, mu, factor, previous, eps=1e-6):
    # TODO: replace this exception with your implementation.
    raise NotImplementedError("Check a solver derivative")

In [ ]:
def reference_gradient_check(layer, mu, factor, previous, eps=1e-6):
    mu = mu.clone().requires_grad_(True)
    weights = layer(mu, factor, previous)
    automatic = torch.autograd.grad(weights[0], mu)[0][0]
    direction = torch.zeros_like(mu)
    direction[0] = eps
    with torch.no_grad():
        finite = (layer(mu + direction, factor, previous)[0]
                  - layer(mu - direction, factor, previous)[0]) / (2 * eps)
    return automatic, finite

if USE_SOLUTIONS:
    gradient_check = reference_gradient_check

In [ ]:
smooth_layer = PortfolioLayer(N, LAM, CAP, cost=0, diagonal=True)
automatic, finite = gradient_check(smooth_layer, probe_mu, reference_factor, previous0)
# Analytical equality-constrained derivative for equal diagonal variances.
analytical = (1 - 1 / N) / (2 * LAM * reference_factor[0].square())
assert torch.allclose(automatic, finite, rtol=.02, atol=.1)
assert torch.allclose(automatic, analytical, rtol=.02, atol=.1)
print({"autograd": automatic.item(), "finite_difference": finite.item(),
       "analytical": analytical.item()})

## 5. The upper problem: learn forecasts that produce useful decisions

Use the predicted standard deviations for **allocation**, but a separate known factor $A_{\rm ref,t}$ for **scoring**:
$$\ell_t=-y_t^\top w_t^*+\lambda\|A_{\rm ref,t}w_t^*\|^2+
\tfrac c2\|w_t^*-w_{t-1}^*\|_1+\beta\operatorname{NLL}_t,\qquad\beta=10^{-3}.$$
The model minimizes the mean of these losses over training dates. A predicted variance can change a decision, but cannot directly lower its scoring risk simply by shrinking itself. The NLL term still directly trains both forecast heads.

The chain rule includes $\partial\ell/\partial w^*$ times $\partial w^*/\partial(\mu,v)$ times $\partial(\mu,v)/\partial\theta$, plus the direct NLL gradient. Pretraining minimizes only NLL and never solves a portfolio problem. A point baseline substitutes MSE pretraining, historical allocation risk and no NLL term.

Weights feed the next date's turnover penalty. Keep that graph within an eight-date batch; detach holdings after an optimizer update. Detaching every date removes inter-date gradient paths; never detaching can reuse a freed graph after a parameter update. The state here is a weight-difference approximation: no return drift, cash ledger or execution simulation is added.

### Exercise: Decision plus statistical loss

Use the supplied reference risk factor, not the predicted variance. Preserve every gradient.

Implement `outer_loss` below. The following cell contains a reference solution; try first before opening it.

In [ ]:
def outer_loss(weights, observed, ref_factor, previous, mu, variance, beta=1e-3):
    # TODO: replace this exception with your implementation.
    raise NotImplementedError("Decision plus statistical loss")

In [ ]:
def reference_outer_loss(weights, observed, ref_factor, previous, mu, variance, beta=1e-3):
    decision = (-weights @ observed + LAM * (ref_factor * weights).square().sum()
                + COST / 2 * (weights - previous).abs().sum())
    return decision + beta * marginal_nll(mu, variance, observed)

if USE_SOLUTIONS:
    outer_loss = reference_outer_loss

In [ ]:
model_probe = GaussianReturnPredictor(F, depth=0).double()
mu, variance = model_probe.predict_moments(X[0])
w = allocator(mu, variance.sqrt(), previous0)
loss = outer_loss(w, y[0], reference_factor, previous0, mu, variance)
expected = allocator.loss(w, y[0], reference_factor, previous0) + .001 * marginal_nll(mu, variance, y[0])
assert torch.allclose(loss, expected)
# Isolate the allocation path, without the direct NLL gradient.
decision = allocator.loss(w, y[0], reference_factor, previous0)
g_mean, g_variance = torch.autograd.grad(decision, (mu, variance), retain_graph=True)
assert all(torch.isfinite(g).all() and g.norm() > 1e-10 for g in (g_mean, g_variance))
loss.backward()
assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in model_probe.parameters())
print("Both mean and variance receive gradients through allocation")

### Exercise: One chronological training/evaluation pass

For E2E, solve, score, and pass weights forward. For NLL, bypass allocation. Average per date, update every eight dates, then detach holdings. Evaluation must not update parameters.

Implement `run_pass` below. The following cell contains a reference solution; try first before opening it.

In [ ]:
def run_pass(model, indices, mode, optimizer=None):
    # TODO: replace this exception with your implementation.
    raise NotImplementedError("One chronological training/evaluation pass")

In [ ]:
def reference_run_pass(model, indices, mode, optimizer=None):
    training = optimizer is not None
    model.train(training)
    previous = previous0.clone()
    total = 0.0
    ids = list(indices)
    with torch.set_grad_enabled(training):
        for start in range(0, len(ids), 8):
            batch = ids[start:start + 8]
            if training:
                optimizer.zero_grad(set_to_none=True)
            losses = []
            for t in batch:
                mu, variance = model.predict_moments(X[t])
                if mode == "nll":
                    loss = marginal_nll(mu, variance, y[t])
                elif mode == "e2e":
                    weights = allocator(mu, variance.sqrt(), previous)
                    loss = outer_loss(weights, y[t], reference_factor, previous, mu, variance)
                    previous = weights
                else:
                    raise ValueError(mode)
                losses.append(loss)
            objective = torch.stack(losses).mean()
            if training:
                objective.backward()
                optimizer.step()
            total += objective.item() * len(batch)
            previous = previous.detach()
    return total / len(ids)

if USE_SOLUTIONS:
    run_pass = reference_run_pass

### A deliberately short worked run

The helper below selects the best validation checkpoint separately in each phase, as the production trainer does. Four epochs per phase illustrate wiring, not convergence. The small linear backbone (`depth=0`) keeps the lesson readable; the production default uses three hidden layers. No hyperparameters are selected on test data. Each independent evaluation starts at equal weights; this is a cold-start test convention, not the production runner's continuous holdings across test windows.

In [ ]:
def fit_phase(model, mode, epochs=4):
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    best, best_state, history = float("inf"), None, []
    for epoch in range(epochs):
        training_loss = run_pass(model, train, mode, optimizer)
        validation_loss = run_pass(model, validation, mode)
        history.append((training_loss, validation_loss))
        if validation_loss < best:
            best, best_state = validation_loss, deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return np.array(history)

torch.manual_seed(7)
model = GaussianReturnPredictor(F, depth=0).double()
pretraining_history = fit_phase(model, "nll")
pretrained = deepcopy(model)
e2e_history = fit_phase(model, "e2e")
for phase, history in [("NLL pretraining", pretraining_history), ("E2E fine-tuning", e2e_history)]:
    plt.figure(figsize=(5, 2))
    plt.plot(history[:, 0], label="train")
    plt.plot(history[:, 1], label="validation")
    plt.title(phase); plt.xlabel("Epoch (starting at zero)")
    plt.ylabel("Phase objective"); plt.legend(); plt.show()
assert np.isfinite(pretraining_history).all() and np.isfinite(e2e_history).all()

## 6. Evaluate without promising an improvement

Compare the NLL-pretrained checkpoint and the E2E checkpoint on untouched dates. Report mean realized return, turnover, reference-risk decision loss, NLL and marginal 95% coverage. Equal weighting is a useful decision comparator. These few dependent portfolio observations cannot establish statistical superiority or calibration. NLL can be negative because it scores a continuous density; values from different phase objectives should not be compared directly. No annualization is applied.

In [ ]:
def evaluate(model=None):
    previous = previous0.clone()
    rows, weights_path = [], []
    with torch.no_grad():
        for t in test:
            if model is None:
                weights = previous0.clone()
                nll, coverage = float("nan"), float("nan")
            else:
                mu, variance = model.predict_moments(X[t])
                weights = allocator(mu, variance.sqrt(), previous)
                nll = marginal_nll(mu, variance, y[t]).item()
                coverage = (((y[t] - mu).abs() <= 1.96 * variance.sqrt()).double().mean().item())
            turnover = (weights - previous).abs().sum() / 2
            rows.append([float(weights @ y[t]), float(turnover),
                         allocator.loss(weights, y[t], reference_factor, previous).item(),
                         nll, coverage])
            weights_path.append(weights.numpy())
            previous = weights
    return np.array(rows), np.array(weights_path)

for name, candidate in [("equal weight", None), ("pretrained", pretrained), ("E2E", model)]:
    scores, weights_path = evaluate(candidate)
    print(name, dict(zip(["mean_return", "mean_turnover", "decision_loss", "NLL", "coverage_95"],
                         scores.mean(axis=0).round(6))))
    assert np.allclose(weights_path.sum(axis=1), 1, atol=1e-6)
plt.plot(weights_path)
plt.xlabel("Test interval"); plt.ylabel("E2E weight"); plt.show()

## 7. Optional Bayesian-style update: what is being updated?

This is a standalone extension in `BLUpdate`, **not used by the active Gaussian trainer**. Let an unknown expected-return vector $m$ have prior $m\sim N(\pi,C)$, where $C=\tau\Sigma_{\rm hist}$. Suppose network views $q$ satisfy $q\mid m\sim N(m,\Omega)$ with independent view errors and positive diagonal $\Omega$. Gaussian conditioning gives
$$\bar m=\pi+C(C+\Omega)^{-1}(q-\pi),\qquad
V=C-C(C+\Omega)^{-1}C.$$
Equivalently $V=(C^{-1}+\Omega^{-1})^{-1}$. Implement linear solves rather than explicit inverses. As $\Omega\to0$, $\bar m\to q$; as view noise grows, $\bar m\to\pi$. In one dimension the weight on the view is $C/(C+\Omega)$.

$V$ is uncertainty about the **mean**; the Gaussian network's $v$ describes conditional **returns**. They are not interchangeable. If future returns satisfy $y\mid m\sim N(m,\Sigma_{\rm return})$ independently of view errors, integrating $m$ gives predictive covariance $\Sigma_{\rm return}+V$. The repository's standalone BL module updates only the mean and deliberately leaves allocation risk unchanged. Neither that module nor this tutorial infers a posterior over neural-network parameters.

A Bayesian reading requires an appropriate prior and likelihood. A learned view extracted from the same data as the prior need not satisfy independence assumptions. Passing forecasts through a Gaussian formula alone does not validate those assumptions.

### Exercise: Fuse prior and views

Return posterior mean and posterior mean-covariance. Use torch.linalg.solve so gradients flow through views.

Implement `bayesian_update` below. The following cell contains a reference solution; try first before opening it.

In [ ]:
def bayesian_update(views, prior, C, omega):
    # TODO: replace this exception with your implementation.
    raise NotImplementedError("Fuse prior and views")

In [ ]:
def reference_bayesian_update(views, prior, C, omega):
    system = C + torch.diag(omega)
    mean = prior + C @ torch.linalg.solve(system, views - prior)
    covariance = C - C @ torch.linalg.solve(system, C)
    return mean, covariance

if USE_SOLUTIONS:
    bayesian_update = reference_bayesian_update

In [ ]:
C = torch.eye(N) * 5e-6
prior = torch.zeros(N)
views = torch.tensor([.004, -.002, .001, .003], requires_grad=True)
omega = torch.full((N,), 1e-5)
posterior, V = bayesian_update(views, prior, C, omega)
assert torch.allclose(posterior, views / 3)
assert torch.linalg.eigvalsh(V).min() > 0
assert torch.allclose(posterior, BLUpdate(omega, tau=.05)(views, prior, C / .05))
assert torch.allclose(bayesian_update(views, prior, C, omega * 1e-8)[0], views, atol=1e-9)
assert torch.allclose(bayesian_update(views, prior, C, omega * 1e8)[0], prior, atol=1e-9)
assert torch.isfinite(torch.autograd.grad(posterior.sum(), views)[0]).all()
print("Posterior mean:", posterior.detach().numpy())

## 8. Explain the thesis without the notebook

Write answers before checking the notes below.

1. State the lower optimization problem, dimensions, constraints and the covariance-factor identity. Why is the factor transposed for dense Cholesky?
2. Derive the equality-constrained KKT derivative. What happens when a bound becomes active? Does convex allocation make neural-network training convex?
3. Trace the gradient from realized utility to each Gaussian head. Identify one `.detach()` that breaks learning and one necessary truncation boundary.
4. Explain why NLL pretraining and E2E fine-tuning optimize different criteria. Why use independent reference risk and retain an NLL term?
5. Distinguish conditional return variance, uncertainty about an unknown expected return and uncertainty about network parameters. Which are actually implemented?
6. Explain the Bayesian-style update and its limits. Is it currently in the production training path?
7. Describe information at decision time, execution timing, chronological validation and label purging. Why can missing realized returns not be silently set to zero?
8. What would need to change for a changing universe? What evidence would you need before making a performance claim?

<details><summary>Answer guide</summary>

1. $w,\mu,w^-\in\mathbb R^N$, $A\in\mathbb R^{N\times N}$ or a diagonal vector. Fully invested, long-only, capped; $A^\top A=\Sigma$, so $A=L^\top$.
2. Differentiate stationarity and feasibility; solve the block system. Bounds change the local active equations and may create nondifferentiable points. The outer network problem remains nonconvex.
3. Outcome → decision loss → optimal weights → predicted mean/SD → network. Detaching forecasts or converting to NumPy breaks this path. Detach holdings after an eight-date optimizer update.
4. NLL scores densities; E2E scores induced decisions plus NLL. Independent reference risk prevents directly rewarding understated scoring risk; NLL keeps distributional learning pressure without guaranteeing calibration.
5. The active single network implements heteroscedastic return uncertainty. The standalone BL formula has a latent-mean interpretation under assumptions. No posterior over network parameters is fitted.
6. Precision-weighted Gaussian conditioning moves the prior toward reliable views. Current `training/e2e.py` bypasses it.
7. Features precede execution; labels end later. Purge labels crossing a training boundary. A missing price is neither a zero economic return nor proof of an executed sale.
8. Align holdings by stable IDs, initialize entrants, confirm exits and settlement returns, validate complete consecutive holding intervals. The repository handles approved aligned exits, not a general pending-order valuation ledger. Larger properly timed out-of-sample experiments, baselines, robustness and uncertainty analysis are needed for performance claims.
</details>

### Suggested experiments after completing the core

Change one quantity at a time: cap, cost, $\lambda$ or NLL weight. Predict what should happen first. Test whether larger cost reduces turnover; this is a hypothesis, not a per-sample guarantee. Try detaching predicted SD and repeat the isolated variance-gradient check. Replace diagonal covariance with $D R D$ using a fixed valid correlation matrix and verify the factor orientation. These extensions are optional; the default example stays small.

### Map to the actual implementation

Paths are relative to the repository root:

| Tutorial concept | Implementation |
|---|---|
| Shared return network, variance transform, Gaussian NLL | `src/btlight/ml/model/return_predictor.py` |
| DPP allocation and realized decision loss | `src/btlight/optimization/differentiable.py` |
| Pretraining, eight-date E2E updates, checkpoint selection | `src/btlight/ml/training/e2e.py` |
| Historical factors and ID-aligned holdings | `src/btlight/ml/training/portfolio_data.py` |
| Standalone prior/view mean update | `src/btlight/estimation/differentiable_bl.py` |

The production runner additionally performs rolling chronological retraining and carries holdings across test windows. This notebook fits one split, uses fixed assets and does not call the full runner or reproduce data cleaning, corporate actions, feature engineering or production performance metrics.

### Further reading

- [CVXPYlayers official repository](https://github.com/cvxpy/cvxpylayers): API and DPP usage. This notebook targets the repository's pinned **0.1.9** API; newer documentation may describe additional backends.
- [Agrawal et al., Differentiable Convex Optimization Layers (2019)](https://arxiv.org/abs/1910.12430): conic solution-map differentiation.
- [CVXPY DPP tutorial](https://www.cvxpy.org/tutorial/dpp/index.html): parameter rules and factorized quadratic forms.
- Local `theory.tex` develops the convexity, KKT and scoring-rule background. Use the implementation paths above to verify which research ideas are active.
